# Build AERA Recovery for POCO Pad 5G (`ruan`)

Run this notebook on a **free Google Colab** runtime (Runtime → Change runtime type → CPU is fine).

It syncs AERA `aera-16.0`, adds the prepared `ruan` device tree from GitHub, builds `recovery.img`, and downloads it. The build can take 2–6 hours; keep the tab open. If a cell fails, copy the **last ~100 lines** of the error and send them back for fixing.

⚠️ Colab free has ~100 GB disk and ~12 GB RAM. This is tight but usually works with `--depth=1` and low build parallelism. Do not flash the result until it has been boot-tested.

In [ ]:
import os, shutil, subprocess, textwrap, time
BASE = '/content/aera'
os.makedirs(BASE, exist_ok=True)
print('disk before:', subprocess.getoutput('df -h /content | tail -1'))
print('mem:', subprocess.getoutput('free -h | head -2'))

In [ ]:
!sudo apt-get update -qq
!sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq git-core gnupg flex bison build-essential zip curl zlib1g-dev libc6-dev-i386 libncurses5 lib32ncurses5-dev x11proto-core-dev libx11-dev lib32z1-dev libgl1-mesa-dev libxml2-utils xsltproc unzip fontconfig python3 openjdk-17-jdk ccache schedtool rsync squashfs-tools dosfstools e2fsprogs > /dev/null
!curl -sSL https://storage.googleapis.com/git-repo-downloads/repo | sudo tee /usr/local/bin/repo > /dev/null && sudo chmod a+x /usr/local/bin/repo
!java -version 2>&1 | head -2

In [ ]:
# Get the prepared ruan device tree
!cd /content && git clone -q --depth=1 https://github.com/xstacy666/AERA-ruan.git ruan-src
!mkdir -p {BASE}/device/xiaomi/ruan
!cp -a /content/ruan-src/. {BASE}/device/xiaomi/ruan/
!rm -rf {BASE}/device/xiaomi/ruan/.git {BASE}/device/xiaomi/ruan/.github
print(subprocess.getoutput('ls ' + BASE + '/device/xiaomi/ruan | head -20'))

In [ ]:
!cd {BASE} && git config --global user.name builder && git config --global user.email builder@example.com
!cd {BASE} && repo init --depth=1 -u https://github.com/AERA-Recovery/android_manifest -b aera-16.0
!mkdir -p {BASE}/.repo/local_manifests
with open(f'{BASE}/.repo/local_manifests/ruan.xml', 'w') as f:
    f.write('''<?xml version="1.0" encoding="UTF-8"?>
<manifest>
  <remote name="AERA" fetch="https://github.com/AERA-Recovery" revision="aera-16.0" />
  <project path="device/xiaomi/sm84xx-common" name="android_device_xiaomi_sm84xx-common-AERA" remote="AERA" />
  <project path="vendor/recovery" name="android_vendor_recovery" remote="AERA" />
</manifest>
''')

In [ ]:
!cd {BASE} && repo sync -c --no-clone-bundle --no-tags -j4 --fail-fast
print('disk after sync:', subprocess.getoutput('df -h /content | tail -1'))

In [ ]:
!cd {BASE} && ALLOW_MISSING_DEPENDENCIES=true USE_CCACHE=1 CCACHE_DIR=/content/.ccache bash -c 'ccache -M 8G; source build/envsetup.sh && lunch twrp_ruan-bp2a-eng && mka -j2 recoveryimage'
print('disk after build:', subprocess.getoutput('df -h /content | tail -1'))

In [ ]:
import os
p = f'{BASE}/out/target/product/ruan/recovery.img'
print('exists:', os.path.exists(p), 'size:', os.path.getsize(p) if os.path.exists(p) else 0)
if os.path.exists(p):
    shutil.copy(p, '/content/recovery.img')
    from google.colab import files
    files.download('/content/recovery.img')
    print('Downloaded recovery.img — send it to the chat!')